In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

import pickle
import numpy as np
from tqdm import tqdm

from src.model.train_and_eval import train_with_parameter

import random
import torch

## Ultra-Strong Coupling (USC) — Full Rabi Model

**Goal:** show that the PINN fails to recover the correct coupling parameters when the system is in the USC regime (g ~ ω_c ~ ω_a).

- Data generated with the **full Rabi Hamiltonian** (`picture="rabi2"`):
  H = ω_c a†a + ω_a σ₊σ₋ + g₁(a†σ₋ + aσ₊) + g₂(aσ₋ + a†σ₊)
- g₁ = g₂ = 1.0, ω_c = ω_a = 1.0  →  g/ω = 1 (deep USC)
- Training learns both g₁ and g₂ as free parameters via `train_with_parameter`
- ODE loss enforces the Schrödinger equation in the interaction picture (`rabi2_ip`)

In [2]:
# --- USC parameters: g ~ omega ---
params = {"wc": 1.0, "wa": 1.0, "g1": 1.0, "g2": 1.0}
tfinal = 2 * np.pi
n_time_steps = 400
init_state = "fock"
dims = {"atom": 2, "field": 4}  # field=4: USC populates higher Fock states
picture = "rabi2"
epochs = 30000
n_points_loss = n_time_steps

In [3]:
# --- Single run ---
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

models_dict, loss_dict = train_with_parameter(
    epochs=epochs,
    params=params,
    tfinal=tfinal,
    n_time_steps=n_time_steps,
    init_state=init_state,
    picture=picture,
    dims=dims,
    n_points_loss=n_points_loss,
    n_parameter=2,  # learns g1 and g2
    is_scaled=False,
)

results_dict = {"models": models_dict, "loss": loss_dict, "params": params}

with open(f"../saved_artifacts/training/results_usc_rabi_{init_state}.pkl", "wb") as f:
    pickle.dump(results_dict, f)

/Users/rodrigosilva/Documents/Mestrado/pinn-effective-hamiltonian/.venv/lib/python3.9/site-packages/qutip/solver/options.py:16: FutureWarning: Dedicated options class are no longer needed, options should be passed as dict to solvers.
  warnings.warn(
100%|██████████| 30000/30000 [04:25<00:00, 112.89it/s]


In [4]:
# --- Multiple runs (optional) ---
n_iter = 5
base_seed = 42

results_dict_multi = {f"run_{i}": {"loss": None, "models": None} for i in range(n_iter)}

for i in tqdm(range(n_iter)):
    print(f"--- RUN {i+1} ---")

    seed = base_seed + i
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

    models_dict, loss_dict = train_with_parameter(
        epochs=epochs,
        params=params,
        tfinal=tfinal,
        n_time_steps=n_time_steps,
        init_state=init_state,
        picture=picture,
        dims=dims,
        n_points_loss=n_points_loss,
        n_parameter=2,
        is_scaled=False,
    )

    results_dict_multi[f"run_{i}"]["loss"] = loss_dict
    results_dict_multi[f"run_{i}"]["models"] = models_dict

with open(f"../saved_artifacts/training/results_usc_rabi_{init_state}_multi.pkl", "wb") as f:
    pickle.dump(results_dict_multi, f)

  0%|          | 0/5 [00:00<?, ?it/s]

--- RUN 1 ---


 20%|██        | 1/5 [04:34<18:16, 274.21s/it]

--- RUN 2 ---


 20%|██        | 1/5 [07:54<31:39, 474.97s/it]


KeyboardInterrupt: 